# 01. Data Cleaning & Normalization Pipeline
**AI Customer Support Ticket Intelligence Platform**

This notebook walks through loading the raw Kaggle customer support tickets dataset, inspecting schema elements, handling missing values, removing duplicates, and performing category mapping and data enrichment (Support Agents, Departments, and synthetic Created/Resolved dates).

In [ ]:
import os
import pandas as pd
import numpy as np
import random
from datetime import datetime

# Paths
raw_csv_path = '../dataset/customer_support_tickets.csv'
cleaned_csv_path = '../dataset/cleaned_customer_support_tickets.csv'

In [ ]:
# Load dataset
df = pd.read_csv(raw_csv_path)
print(f"Initial dataset shape: {df.shape}")
df.head(3)

### 1. Remove Duplicates & Clean Column Names

In [ ]:
# Drop duplicate tickets based on 'Ticket ID'
df = df.drop_duplicates(subset=['Ticket ID'])
print(f"Shape after dropping duplicates: {df.shape}")

# Standardize column names (replace spaces with underscores)
df.columns = [col.replace(' ', '_') for col in df.columns]
print("Standardized Column names:", df.columns.tolist())

### 2. Handle Missing Values

In [ ]:
print("Missing values sum before:")
print(df.isnull().sum())

# Fill Resolution text
df['Resolution'] = df['Resolution'].fillna('Under Investigation')

# Fill Satisfaction rating with -1 (Unrated)
df['Customer_Satisfaction_Rating'] = df['Customer_Satisfaction_Rating'].fillna(-1).astype(int)

print("\nMissing values sum after cleaning:")
print(df[['Resolution', 'Customer_Satisfaction_Rating']].isnull().sum())

### 3. Category & Priority Mapping
* Map ticket priorities (Group `Critical` and `High` together for standard class labeling)
* Align categories: Classify ticket descriptions using keyword mappings into `Technical`, `Billing`, `Refund`, `Shipping`, `Account`, `Login` categories.

In [ ]:
# Map priority to standard Low, Medium, High
priority_map = {'Low': 'Low', 'Medium': 'Medium', 'High': 'High', 'Critical': 'High'}
df['Priority'] = df['Ticket_Priority'].map(priority_map)
print("Priority distribution:")
print(df['Priority'].value_counts())

# Keyword categorization logic
def assign_category(row):
    subject = str(row['Ticket_Subject']).lower()
    desc = str(row['Ticket_Description']).lower()
    text = subject + " " + desc
    
    if any(kw in text for kw in ['refund', 'money back', 'chargeback', 'double charge', 'reimbursement']):
        return 'Refund'
    elif any(kw in text for kw in ['bill', 'invoice', 'payment', 'credit card', 'price', 'subscription', 'charge']):
        return 'Billing'
    elif any(kw in text for kw in ['shipping', 'delivery', 'shipped', 'track', 'mail', 'transit', 'post', 'courier']):
        return 'Shipping'
    elif any(kw in text for kw in ['login', 'password', 'reset', 'signin', 'signout', 'credentials', 'access', 'auth']):
        return 'Login'
    elif any(kw in text for kw in ['account', 'profile', 'member', 'cancel account', 'subscription cancel', 'deactivate']):
        return 'Account'
    else:
        return 'Technical'

df['Ticket_Category'] = df.apply(assign_category, axis=1)
print("\nTicket category distribution:")
print(df['Ticket_Category'].value_counts())

### 4. Agent Enrichment & Synthetic Time Spans

In [ ]:
# Setup Support Agents list
agents = ["Alice Smith", "Bob Johnson", "Charlie Brown", "Diana Prince", "Ethan Hunt", 
          "Fiona Gallagher", "George Clark", "Hannah Abbott", "Ian Malcolm", "Julia Roberts"]
random.seed(42)
df['Support_Agent'] = [random.choice(agents) for _ in range(len(df))]

# Generate clean Created_Date and Resolved_Date
np.random.seed(42)
start_time = datetime(2022, 1, 1).timestamp()
end_time = datetime(2024, 12, 31).timestamp()
random_created = np.random.uniform(start_time, end_time, len(df))

df['Created_Date'] = pd.to_datetime(random_created, unit='s')

# Calculate resolution status details
df['Status'] = df['Ticket_Status'].fillna('Open')

resolved_dates = []
resolution_times = []

for idx, row in df.iterrows():
    if row['Status'] == 'Closed':
        hours = round(random.uniform(2.0, 120.0), 2)
        res_date = row['Created_Date'] + pd.to_timedelta(hours, unit='h')
        resolved_dates.append(res_date)
        resolution_times.append(hours)
    else:
        resolved_dates.append(pd.NaT)
        resolution_times.append(np.nan)

df['Resolved_Date'] = pd.to_datetime(resolved_dates)
df['Resolution_Time'] = resolution_times

### 5. Export Cleaned Dataset

In [ ]:
# Save to dataset folder
df.to_csv(cleaned_csv_path, index=False)
print(f"Pipeline completed. Cleaned dataset size: {df.shape}. Saved to {cleaned_csv_path}")